# สำรวจข้อมูล 4/4: ตอบคำถาม และสรุป Insight

| ไฟล์ | เกณฑ์ 3.5.2 (18 คะแนน) |
|---|---|
| ✅ eda_step1_descriptive | 3.5.2.1 สถิติเบื้องต้น (8) |
| ✅ eda_step2_trends | 3.5.2.2 แนวโน้ม/ความสัมพันธ์ (5) |
| ✅ eda_step3_outliers_missing | 3.5.2.3 ค่าผิดปกติ/การขาดหาย (3) |
| **→** eda_step4_insights | 3.5.2.4 Insight + ตอบคำถามจาก step1_questions (2) |

ไฟล์นี้ **ปิดวงจรการวิเคราะห์**: กลับไปตอบคำถาม 4 ข้อที่ตั้งไว้ใน `step1_questions.ipynb` ด้วยตัวเลขจริง แล้วสรุปเป็น Insight ที่นำไปใช้ตัดสินใจได้

ตอบเกณฑ์ 3.5.2.4 (2 คะแนน):
- Bullet สรุป 3–5 ข้อ ที่ตอบโจทย์และบอก "แล้วทำอะไรต่อ"
- มีตัวเลขอ้างอิง (Δ%, mean±std, rate) และข้อจำกัดอย่างน้อย 1 ข้อ

ผลลัพธ์ `eda_output/insights.md` ใช้ทำสไลด์และใส่ในแดชบอร์ดได้ทันที

> รัน **Kernel → Restart Kernel and Run All Cells**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DATA = os.path.join("final", "energy_clean.csv")
if not os.path.exists(DATA):
    raise FileNotFoundError("ไม่พบ final/energy_clean.csv ให้รันขั้นทำความสะอาด 1-5 ให้สำเร็จก่อน")
OUT = "eda_output"                       # เก็บรูปและตารางไว้ใช้ทำสไลด์
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(DATA)
LATEST = int(df.loc[df["year_complete"], "year"].max())   # ปีล่าสุดที่ข้อมูลครบ = 2021
y = df[df["year"] == LATEST].copy()                        # ภาพตัดขวาง (ทุกประเทศ ปีเดียว)
ASEAN = sorted(df.loc[df["is_asean"], "country"].unique())
FUEL_COLOR = {"coal": "#4A4039", "oil": "#8C5A3C", "gas": "#E0913A", "nuclear": "#7B61A8", "hydro": "#2E78AD",
              "wind": "#5FB3A1", "solar": "#E8B923", "biofuel": "#7A9A3A", "other_renewables": "#9AA5A8"}

def savefig(name):
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")

print(f"ข้อมูล: {df.shape[0]:,} แถว x {df.shape[1]} คอลัมน์ | {df['country'].nunique()} ประเทศ | ปี {df['year'].min()}-{df['year'].max()}")
print(f"ปีที่ใช้วิเคราะห์ภาพตัดขวาง: {LATEST} ({len(y)} ประเทศ)")

In [ ]:
th = df[df["country"] == "Thailand"].set_index("year").sort_index()
answers = {}      # เก็บคำตอบไว้สรุปท้ายไฟล์

## คำถามที่ 1: ความต้องการไฟฟ้าของไทยเพิ่มขึ้นกี่เปอร์เซ็นต์ ระหว่างปี 2000–2022
ใช้สูตร **% เปลี่ยนแปลง = (ค่าปลาย − ค่าต้น) ÷ ค่าต้น × 100** และ **อัตราโตเฉลี่ยต่อปี (CAGR) = (ค่าปลาย ÷ ค่าต้น)^(1/จำนวนปี) − 1**

In [ ]:
d0, d1 = th.loc[2000, "electricity_demand"], th.loc[2022, "electricity_demand"]
d21 = th.loc[LATEST, "electricity_demand"]
pct = (d1 / d0 - 1) * 100
cagr = ((d1 / d0) ** (1 / 22) - 1) * 100
print(f"ความต้องการไฟฟ้าไทย: {d0:.1f} TWh (2000) -> {d1:.1f} TWh (2022)")
print(f"เพิ่มขึ้น {pct:.0f}% | โตเฉลี่ย {cagr:.1f}% ต่อปี")
print(f"ตรวจด้วยปีที่ข้อมูลครบ (2000-{LATEST}): +{(d21 / d0 - 1) * 100:.0f}%  (ปี 2022 เป็นข้อมูลเบื้องต้น)")
answers["Q1"] = f"ความต้องการไฟฟ้าไทยเพิ่มขึ้น {pct:.0f}% จาก {d0:.0f} เป็น {d1:.0f} TWh (2000–2022) หรือโตเฉลี่ย {cagr:.1f}% ต่อปี"

ax = th["electricity_demand"].plot(figsize=(9, 3.5), marker="o", color="#1C2733", title="Thailand electricity demand (TWh)")
ax.set_ylim(0, None); ax.annotate(f"+{pct:.0f}%", (2022, d1), xytext=(-40, -25), textcoords="offset points", fontsize=12, color="#B3412E")
plt.tight_layout(); savefig("q1_demand.png"); plt.show()

## คำถามที่ 2: สัดส่วนพลังงานหมุนเวียนในการผลิตไฟฟ้าของไทยเปลี่ยนไปอย่างไร
**ข้อจำกัด:** คำถามเดิมตั้งช่วง 1990–2022 แต่ข้อมูลที่ทำความสะอาดแล้วเริ่มปี 2000 (เพราะก่อนปี 2000 มีข้อมูลไฟฟ้าแค่ราว 1 ใน 3 ของประเทศ) จึงตอบช่วง **2000–2022** และระบุไว้เป็นข้อจำกัด

In [ ]:
REN = ["hydro", "biofuel", "solar", "wind", "other_renewable_exc_biofuel"]
share_col = lambda f: "other_renewables_share_elec_exc_biofuel" if f == "other_renewable_exc_biofuel" else f"{f}_share_elec"
r0, r1 = th.loc[2000, "renewables_share_elec"], th.loc[2022, "renewables_share_elec"]
contrib = pd.Series({f: th.loc[2022, share_col(f)] - th.loc[2000, share_col(f)] for f in REN}).sort_values(ascending=False)
print(f"สัดส่วนหมุนเวียนไทย: {r0:.1f}% (2000) -> {r1:.1f}% (2022) = +{r1 - r0:.1f} จุด (เพิ่มขึ้น {r1 / r0:.1f} เท่า)")
print("ส่วนที่เพิ่มมาจาก (จุด %):"); print(contrib.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 3.8))
bottom = np.zeros(len(th))
for f, col in zip(REN, ["#2E78AD", "#7A9A3A", "#E8B923", "#5FB3A1", "#9AA5A8"]):
    v = th[share_col(f)].fillna(0).values
    ax.bar(th.index, v, bottom=bottom, color=col, label=f.replace("_exc_biofuel", "")); bottom += v
ax.set_title("Thailand: renewables share of electricity by source (%)"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); savefig("q2_renewables.png"); plt.show()

# เทียบกับโลกช่วงเดียวกัน
w = df[df["year_complete"]].groupby("year")[["renewables_electricity", "electricity_generation"]].sum()
w_share = w["renewables_electricity"] / w["electricity_generation"] * 100
print(f"\nเทียบโลก 2000->{LATEST}: {w_share.loc[2000]:.1f}% -> {w_share.loc[LATEST]:.1f}% (+{w_share.loc[LATEST] - w_share.loc[2000]:.1f} จุด)")
print(f"ไทย 2000->{LATEST}: +{th.loc[LATEST, 'renewables_share_elec'] - r0:.1f} จุด")
answers["Q2"] = (f"สัดส่วนหมุนเวียนไทยเพิ่มจาก {r0:.1f}% เป็น {r1:.1f}% (2000–2022) มาจากชีวมวล (+{contrib['biofuel']:.1f} จุด) "
                 f"แสงอาทิตย์ (+{contrib['solar']:.1f}) และลม (+{contrib['wind']:.1f}) ขณะที่พลังน้ำลดลง ({contrib['hydro']:.1f}) "
                 f"แต่ยังต่ำกว่าโลก ({w_share.loc[LATEST]:.1f}% ในปี {LATEST})")

## คำถามที่ 3: ไทยพึ่งก๊าซธรรมชาติผลิตไฟฟ้ามากกว่าเพื่อนบ้านอาเซียนหรือไม่ (2010–2022)
เทียบ 5 ประเทศตามคำถามเดิม และดูทั้งอาเซียน 10 ประเทศประกอบ

In [ ]:
Q3 = ["Thailand", "Vietnam", "Malaysia", "Indonesia", "Philippines"]
g = df[df["country"].isin(Q3) & df["year"].between(2010, 2022)].pivot(index="year", columns="country", values="gas_share_elec")
stat = pd.DataFrame({"mean": g.mean(), "std": g.std(), "min": g.min(), "max": g.max(),
                     f"{LATEST}": g.loc[LATEST]}).sort_values("mean", ascending=False).round(1)
rank_th = g.rank(axis=1, ascending=False)["Thailand"]
print(f"ไทยอันดับ 1 (ใช้ก๊าซมากสุดใน 5 ประเทศ) {int((rank_th == 1).sum())} จาก {len(rank_th)} ปี")
display(stat)

fig, ax = plt.subplots(figsize=(9, 4))
for c in Q3:
    ax.plot(g.index, g[c], marker="o", lw=3 if c == "Thailand" else 1.5, label=c,
            color="#1F5F8B" if c == "Thailand" else None, alpha=1 if c == "Thailand" else 0.6)
ax.set_ylim(0, 100); ax.set_title("Gas share of electricity, 2010-2022 (%)"); ax.legend()
plt.tight_layout(); savefig("q3_gas.png"); plt.show()

th_mean, th_std = stat.loc["Thailand", "mean"], stat.loc["Thailand", "std"]
others_mean = stat.drop("Thailand")["mean"].mean()
answers["Q3"] = (f"ใช่ ไทยพึ่งก๊าซมากที่สุดใน 5 ประเทศทุกปี (เฉลี่ย {th_mean:.1f} ± {th_std:.1f}% ช่วง 2010–2022) "
                 f"สูงกว่าค่าเฉลี่ยของอีก 4 ประเทศ ({others_mean:.1f}%) ราว {th_mean / others_mean:.1f} เท่า "
                 f"และในปี {LATEST} ทั้งอาเซียนมีแค่สิงคโปร์ที่ใช้ก๊าซมากกว่าไทย")

## คำถามที่ 4: การใช้ไฟฟ้าต่อคนของไทยสัมพันธ์กับ GDP หรือไม่
**ข้อจำกัด:** GDP มีถึงปี 2018 จึงใช้ช่วง 2000–2018 (19 ปี)

**ข้อควรระวัง:** ตัวแปรที่ **โตตามเวลาทั้งคู่** จะมีสหสัมพันธ์สูงเสมอ แม้ไม่เกี่ยวกันจริง (spurious correlation) จึงตรวจ 2 ชั้น
1. สหสัมพันธ์ของ **ระดับ** (ค่าจริง)
2. สหสัมพันธ์ของ **อัตราโตรายปี** (ตัดผลของแนวโน้มออก) ถ้ายังสัมพันธ์กัน จึงมั่นใจได้มากขึ้น

In [ ]:
t = th.loc[2000:2018].copy()
t["gdp_pc"] = t["gdp"] / t["population"]
r_level = t[["gdp_pc", "per_capita_electricity"]].corr().iloc[0, 1]
gr = t[["gdp_pc", "per_capita_electricity"]].pct_change().dropna() * 100
r_growth = gr.corr().iloc[0, 1]
elasticity = np.polyfit(np.log(t["gdp_pc"]), np.log(t["per_capita_electricity"]), 1)[0]
print(f"r (ระดับ)        = {r_level:.2f}")
print(f"r (อัตราโตรายปี) = {r_growth:.2f}  (n = {len(gr)} ปี)")
print(f"ความยืดหยุ่น: GDP ต่อคน +1% -> ไฟฟ้าต่อคน +{elasticity:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(t["gdp_pc"], t["per_capita_electricity"], color="#1F5F8B")
for yr in [2000, 2009, 2018]:
    ax[0].annotate(str(yr), (t.loc[yr, "gdp_pc"], t.loc[yr, "per_capita_electricity"]), xytext=(5, -10), textcoords="offset points")
ax[0].set_xlabel("GDP per person (international-$)"); ax[0].set_ylabel("Electricity per person (kWh)")
ax[0].set_title(f"Levels: r = {r_level:.2f}")
ax[1].scatter(gr["gdp_pc"], gr["per_capita_electricity"], color="#E0913A")
ax[1].axhline(0, color="grey", lw=0.8); ax[1].axvline(0, color="grey", lw=0.8)
ax[1].set_xlabel("GDP per person growth (%/yr)"); ax[1].set_ylabel("Electricity per person growth (%/yr)")
ax[1].set_title(f"Yearly growth: r = {r_growth:.2f}")
plt.tight_layout(); savefig("q4_gdp.png"); plt.show()
answers["Q4"] = (f"สัมพันธ์กันชัด: r = {r_level:.2f} (ระดับ) และ r = {r_growth:.2f} (อัตราโตรายปี) ช่วง 2000–2018 "
                 f"GDP ต่อคนเพิ่ม 1% ไฟฟ้าต่อคนเพิ่มราว {elasticity:.1f}%")

**อ่านผล:** สหสัมพันธ์ระดับสูงมาก ซึ่งคาดได้เพราะทั้งคู่โตตามเวลา แต่ที่สำคัญคือ **อัตราโตรายปีก็ยังสัมพันธ์กันในทิศทางบวก** ปีที่เศรษฐกิจโตดี การใช้ไฟก็โตตาม
**ข้อจำกัด:** มีแค่ 18 คู่ข้อมูล และสหสัมพันธ์ไม่ได้พิสูจน์ว่า GDP เป็น "สาเหตุ"

## สรุปคำตอบ 4 ข้อ

In [ ]:
ans = pd.DataFrame({"คำตอบ": answers})
ans.index = ["1. ความต้องการไฟฟ้าเพิ่มกี่ %", "2. สัดส่วนหมุนเวียนเปลี่ยนอย่างไร",
             "3. ไทยพึ่งก๊าซมากกว่าเพื่อนบ้านไหม", "4. ไฟฟ้าต่อคนสัมพันธ์กับ GDP ไหม"]
with pd.option_context("display.max_colwidth", 300):
    display(ans)

## Insight ที่นำไปใช้ได้จริง (เกณฑ์ 3.5.2.4)
แต่ละข้อมีรูปแบบ **ข้อค้นพบ (ตัวเลข) → ความหมาย → ข้อเสนอ (ใคร/เมื่อไร)** (ข้อเสนอเป็นตัวอย่าง ให้ทีมปรับตามความเห็นของตัวเอง) สร้างจากตัวเลขที่คำนวณในไฟล์นี้ เมื่อข้อมูลเปลี่ยน ข้อความจะเปลี่ยนตาม

In [ ]:
th21 = th.loc[LATEST]
vn = df[(df["country"] == "Vietnam")].set_index("year")
imp_share = th21["net_elec_imports"] / th21["electricity_demand"] * 100

insights = [
    ("ไทยพึ่งก๊าซธรรมชาติสูงเกินเพื่อนบ้าน",
     f"ก๊าซผลิตไฟฟ้า {th21['gas_share_elec']:.1f}% ของไทย (ปี {LATEST}) สูงเป็นอันดับ 2 ในอาเซียน และเฉลี่ย {th_mean:.1f}±{th_std:.1f}% ตลอด 2010–2022",
     "ค่าไฟไทยผูกกับราคาก๊าซโลก เมื่อราคาก๊าซขึ้น ค่าไฟขึ้นตามทันที",
     "กระทรวงพลังงาน/กกพ.: ตั้งเป้าลดสัดส่วนก๊าซในแผน PDP รอบถัดไป (ภายใน 5 ปี)"),
    ("ความต้องการไฟฟ้าโตต่อเนื่อง แม้ชะลอลง",
     f"ความต้องการเพิ่ม {pct:.0f}% (2000–2022) โตเฉลี่ย {cagr:.1f}%/ปี และไฟฟ้าต่อคนโตเร็วกว่า GDP ต่อคน (ความยืดหยุ่น {elasticity:.1f})",
     "ถ้าเศรษฐกิจโต 3%/ปี ความต้องการไฟฟ้าจะโตเร็วกว่านั้น ต้องเตรียมกำลังผลิตเพิ่ม",
     "กฟผ.: วางแผนกำลังผลิตใหม่จากพลังงานสะอาด แทนการเพิ่มโรงไฟฟ้าก๊าซ (แผน 10 ปี)"),
    ("พลังงานหมุนเวียนไทยโตช้ากว่าโลกและเพื่อนบ้าน",
     f"ไทยเพิ่ม {th.loc[LATEST, 'renewables_share_elec'] - r0:.1f} จุด (2000–{LATEST}) ขณะที่โลกเพิ่ม {w_share.loc[LATEST] - w_share.loc[2000]:.1f} จุด "
     f"และเวียดนามเพิ่มแสงอาทิตย์จาก {vn.loc[2018, 'solar_share_elec']:.1f}% เป็น {vn.loc[LATEST, 'solar_share_elec']:.1f}% ในแค่ 3 ปี",
     "เพื่อนบ้านที่มีทรัพยากรคล้ายกันเปลี่ยนได้เร็วกว่ามาก แปลว่าไทยทำได้ ถ้ามีนโยบายสนับสนุน",
     "ภาครัฐ: ศึกษามาตรการ feed-in tariff ของเวียดนาม และขยายโซลาร์บนหลังคา (1–3 ปี)"),
    ("ไทยพึ่งไฟฟ้านำเข้าเพิ่มขึ้น",
     f"ปี {LATEST} นำเข้าสุทธิ {th21['net_elec_imports']:.1f} TWh = {imp_share:.1f}% ของความต้องการ (ปี 2000 แค่ {th.loc[2000, 'net_elec_imports'] / th.loc[2000, 'electricity_demand'] * 100:.1f}%)",
     "ความมั่นคงด้านไฟฟ้าผูกกับเขื่อนในลาว ซึ่งขึ้นกับปริมาณฝน",
     "กฟผ.: กระจายแหล่งนำเข้าและเพิ่มกำลังผลิตสำรองในประเทศ (ทบทวนทุกปี)"),
]
limitations = [
    "GDP มีถึงปี 2018 และปี 2022 เป็นข้อมูลเบื้องต้น",
    "ข้อมูลรายปีระดับประเทศ ไม่เห็นความต่างรายภูมิภาคหรือรายเดือนในไทย",
    "สหสัมพันธ์ไม่ได้พิสูจน์เหตุและผล",
]
lines = [f"# Insight จากการวิเคราะห์ข้อมูลไฟฟ้า (ข้อมูลถึงปี {LATEST}, ปี 2022 เบื้องต้น)\n"]
for i, (title, what, so, now) in enumerate(insights, 1):
    lines += [f"## {i}. {title}", f"- **ข้อค้นพบ:** {what}", f"- **ความหมาย:** {so}", f"- **ข้อเสนอ:** {now}", ""]
lines += ["## ข้อจำกัด"] + [f"- {l}" for l in limitations]
lines += ["", "## คำตอบคำถามจาก step1_questions"] + [f"- {k}: {v}" for k, v in answers.items()]
text = "\n".join(lines)
with open(os.path.join(OUT, "insights.md"), "w", encoding="utf-8") as f:
    f.write(text)

from IPython.display import Markdown
Markdown(text)

## ✅ สรุปการสำรวจข้อมูลทั้ง 4 ไฟล์ (เกณฑ์ 3.5.2 = 18 คะแนน)
| เกณฑ์ | ไฟล์ | หลักฐานใน `eda_output/` |
|---|---|---|
| 3.5.2.1 สถิติเบื้องต้น (8) | eda_step1 | stats_2021.csv, dist_key_vars.png, category_balance.png |
| 3.5.2.2 แนวโน้ม/ความสัมพันธ์ (5) | eda_step2 | trend_*.png, corr_heatmap.png, compare_groups.png, facet_*.png |
| 3.5.2.3 ค่าผิดปกติ/การขาดหาย (3) | eda_step3 | outliers_2021.csv, outlier_boxplots.png, missing_*.png, co_missing.png |
| 3.5.2.4 Insight (2) | eda_step4 | insights.md, q1–q4 *.png |

**ขั้นถัดไป:** เอกสารออกแบบแดชบอร์ด (ผู้ใช้เป้าหมาย, โครงร่าง) และอินโฟกราฟิกคู่มือการใช้งาน